In [ ]:
import os

!pip install tensorflow matplotlib
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Struktur folder (contoh):
# data_tiny/
# train/
# class0/ ...jpg
# class1/ ...jpg
# val/
# class0/ ...jpg
# class1/ ...jpg

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128,128,3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Generator (harap siapkan folder data_tiny sesuai struktur di atas)
# Using TRAIN_DIR and TEST_DIR from configuration
train_dir = TRAIN_DIR
val_dir = TEST_DIR # Assuming TEST_DIR can be used as validation directory, or adjust as needed

if os.path.exists(train_dir) and os.path.exists(val_dir):
    datagen = ImageDataGenerator(rescale=1./255)
    train_gen = datagen.flow_from_directory(train_dir, target_size=(128,128), batch_size=16)
    val_gen = datagen.flow_from_directory(val_dir, target_size=(128,128), batch_size=16)
    model.fit(train_gen, epochs=3, validation_data=val_gen)
else:
    print(f"Siapkan dataset kustom di folder {train_dir} dan {val_dir} untuk menjalankan training.")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


NameError: name 'TRAIN_DIR' is not defined

In [ ]:
print('Listing contents of /kaggle/input/ (if applicable):')
!ls -F /kaggle/input/

In [ ]:
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
import numpy as np
import pathlib
import datetime
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# 📌 CONFIGURATION - MODIFY THESE SETTINGS AS NEEDED
# ============================================================================
# Available models: 'AlexNet', 'VGG16', 'VGG19', 'ResNet50', 'ResNet101',
# 'MobileNetV2', 'MobileNetV3', 'EfficientNetB0', 'EfficientNetB3',
# 'InceptionV3', 'DenseNet121', 'Xception', 'NASNetMobile'
MODEL_NAME = 'AlexNet'
# Training hyperparameters
BATCH_SIZE = 64
EPOCHS = 150 # Increased to match downloaded notebook
LEARNING_RATE = 0.01 # Good for SGD with AlexNet
OPTIMIZER = 'sgd' # SGD works well for AlexNet
# Data augmentation settings
# NOTE: Set to False for faster initial training
# Set to True for better generalization but slower convergence
USE_AUGMENTATION = False
AUGMENTATION_CONFIG = {
'rotation_range': 20,
'width_shift_range': 0.2,
'height_shift_range': 0.2,
'horizontal_flip': True,
'zoom_range': 0.15,
'shear_range': 0.1,
'brightness_range': [0.8, 1.2],
'fill_mode': 'nearest'
}
# Model fine-tuning settings (NOT applicable for AlexNet - trained from scratch)
FREEZE_BASE = True
UNFREEZE_LAYERS = 20
USE_FINE_TUNING = True
FINE_TUNE_EPOCHS = 20
FINE_TUNE_LR = 0.001
# Regularization
DROPOUT_RATE = 0.0 # Set to 0 to as of now
USE_BATCH_NORM = False
# Callbacks
USE_EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 5 # Reduced patience
USE_LR_SCHEDULER = True
USE_MODEL_CHECKPOINT = True
# Paths
ROOT_DIR = '/content/drive/MyDrive/path/to/your/emotion-detection-fer/' # <<< YOU MUST UPDATE THIS PATH
TRAIN_DIR = os.path.join(ROOT_DIR, 'train/')
TEST_DIR = os.path.join(ROOT_DIR, 'test/')
MODEL_SAVE_PATH = f'best_{MODEL_NAME.lower()}_emotion_model.keras'
# Number of classes (will be auto-detected from data)
NUM_CLASSES = None # Set to None for auto-detection
print(f"✅ Configuration loaded for: {MODEL_NAME}")

In [ ]:
# Use paths from configuration
train_dir = TRAIN_DIR
test_dir = TEST_DIR
# Auto-detect number of classes
if NUM_CLASSES is None:
    NUM_CLASSES = len(os.listdir(train_dir))
    print(f"📊 Auto-detected {NUM_CLASSES} classes: {os.listdir(train_dir)}")

In [ ]:
print('Listing contents of your Google Drive MyDrive to help find your dataset:')
!ls -F '/content/drive/MyDrive/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print('Listing contents of your Google Drive MyDrive to help find your dataset:')
!ls -F '/content/drive/MyDrive/'

In [ ]:
# Data Visualization
train_count=[]
test_count=[]
fig, axes = plt.subplots(1, 7, figsize=(20,8))
for i,j in enumerate(os.listdir(train_dir)):
train_path = os.path.join(train_dir, j)
train_count.append(len(os.listdir(train_path)))
test_path = os.path.join(test_dir, j)
test_count.append(len(os.listdir(test_path)))
img = cv2.imread(os.path.join(train_path, os.listdir(train_path)[0]))
axes[i].imshow(img)
axes[i].set_title(j)
axes[i].axis("off")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].pie(train_count, labels=os.listdir(train_dir), autopct='%1.1f%%',shadow=True, startangle=90)
axes[1].pie(test_count, labels=os.listdir(test_dir), autopct='%1.1f%%',shadow=True, startangle=90)
axes[0].set_title('Train')
axes[1].set_title('Test')
plt.show()

In [ ]:
# Data Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Data generators with optional augmentation
if USE_AUGMENTATION:
  train_idg = ImageDataGenerator(
      rescale=1./255,
      **AUGMENTATION_CONFIG
  )
  print("✅ Data augmentation enabled")
else:
    train_idg = ImageDataGenerator(rescale=1./255)
    print("ℹ️ Data augmentation disabled")
# Test/validation data should not be augmented
test_idg = ImageDataGenerator(rescale=1./255)

In [ ]:
# Model-specific input sizes
MODEL_INPUT_SIZES = {
'AlexNet': (227, 227),
'VGG16': (224, 224),
'VGG19': (224, 224),
'ResNet50': (224, 224),
'ResNet101': (224, 224),
'MobileNetV2': (224, 224),
'MobileNetV3': (224, 224),
'EfficientNetB0': (224, 224),
'EfficientNetB3': (300, 300),
'EfficientNetB7': (600, 600),
'InceptionV3': (299, 299),
'DenseNet121': (224, 224),
'Xception': (299, 299),
'NASNetMobile': (224, 224),
'NASNetLarge': (331, 331),
}
# Get input size for selected model
IMG_HEIGHT, IMG_WIDTH = MODEL_INPUT_SIZES.get(MODEL_NAME, (224, 224))
print(f"📐 Using input size: {IMG_HEIGHT}x{IMG_WIDTH} for {MODEL_NAME}")
# Data generator arguments
class_names = sorted(os.listdir(train_dir))
arg_train = {
'target_size': (IMG_HEIGHT, IMG_WIDTH),
'batch_size': BATCH_SIZE,
'shuffle': True,
'classes': class_names,
'class_mode': 'categorical'
}
arg_test = {
'target_size': (IMG_HEIGHT, IMG_WIDTH),
'batch_size': BATCH_SIZE,
'shuffle': False, # Don't shuffle for evaluation
'classes': class_names,
'class_mode': 'categorical'
}
train = train_idg.flow_from_directory(directory=train_dir, **arg_train)
valid = test_idg.flow_from_directory(directory=test_dir, **arg_test)
print(f"\n📁 Class mapping: {train.class_indices}")

In [ ]:
# Import Keras Layers & Models
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (Dense, Flatten, Conv2D, MaxPooling2D,
Dropout, GlobalAveragePooling2D,
BatchNormalization, Input)
from tensorflow.keras.applications import (
VGG16, VGG19, ResNet50, ResNet101,
MobileNetV2, MobileNetV3Large,
EfficientNetB0, EfficientNetB3, EfficientNetB7,
InceptionV3, DenseNet121, Xception, NASNetMobile, NASNetLarge
)
from tensorflow.keras.optimizers import Adam, SGD, Adamax, RMSprop

In [ ]:
# ============================================================================
# 🏗️ MODEL FACTORY - Supports multiple transfer learning architectures
# ============================================================================
class AlexNet(Sequential):
  """Custom AlexNet implementation"""
  def __init__(self, input_shape, num_classes):
      super().__init__()
      self.add(Conv2D(96, kernel_size=(11,11), strides=4, padding='valid',
                      activation='relu', input_shape=input_shape, kernel_initializer='he_normal'))
      self.add(MaxPooling2D(pool_size=(3,3), strides=(2,2), padding='valid'))
      self.add(Conv2D(256, kernel_size=(5,5), strides=1, padding='same',
                      activation='relu', kernel_initializer='he_normal'))
      self.add(MaxPooling2D(pool_size=(3,3), strides=(2,2), padding='valid'))
      self.add(Conv2D(384, kernel_size=(3,3), strides=1, padding='same',
                      activation='relu', kernel_initializer='he_normal'))
      self.add(Conv2D(384, kernel_size=(3,3), strides=1, padding='same',
                      activation='relu', kernel_initializer='he_normal'))
      self.add(Conv2D(256, kernel_size=(3,3), strides=1, padding='same',
                      activation='relu', kernel_initializer='he_normal'))
      self.add(MaxPooling2D(pool_size=(3,3), strides=(2,2), padding='valid'))
      self.add(Flatten())
      self.add(Dense(4096, activation='relu'))
      self.add(Dropout(DROPOUT_RATE))
      self.add(Dense(4096, activation='relu'))
      self.add(Dropout(DROPOUT_RATE))
      self.add(Dense(1000, activation='relu'))
      self.add(Dense(num_classes, activation='softmax'))
def create_transfer_model(model_name, input_shape, num_classes, freeze_base=True):
    """
    Factory function to create transfer learning models
    Args:
        model_name: Name of the pretrained model
        input_shape: Input shape tuple (height, width, channels)
        num_classes: Number of output classes
        freeze_base: Whether to freeze pretrained layers
    Returns:
        Compiled Keras model
    """
# Map model names to Keras applications
base_models = {
    'VGG16': VGG16,
    'VGG19': VGG19,
    'ResNet50': ResNet50,
    'ResNet101': ResNet101,
    'MobileNetV2': MobileNetV2,
    'MobileNetV3': MobileNetV3Large,
    'EfficientNetB0': EfficientNetB0,
    'EfficientNetB3': EfficientNetB3,
    'EfficientNetB7': EfficientNetB7,
    'InceptionV3': InceptionV3,
    'DenseNet121': DenseNet121,
    'Xception': Xception,
    'NASNetMobile': NASNetMobile,
    'NASNetLarge': NASNetLarge,
}
# Handle custom AlexNet
if model_name == 'AlexNet':
    return AlexNet(input_shape, num_classes)
if model_name not in base_models:
    raise ValueError(f"Unknown model: {model_name}. Available: {list(base_models.keys()) + ['AlexNet']}")
# Load pretrained base model
base_model = base_models[model_name](
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)
# Freeze base model layers if requested
base_model.trainable = not freeze_base
# Build SIMPLIFIED classification head (less regularization)
inputs = Input(shape=input_shape)
x = base_model(inputs, training=not freeze_base) # Use training=True when not frozen
x = GlobalAveragePooling2D()(x)
# Simpler head - just one dense layer before output
x = Dense(256, activation='relu')(x)
x = Dropout(DROPOUT_RATE)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)
print(f"\n📊 Model Summary:")
print(f" Base Model: {model_name}")
print(f" Total layers: {len(model.layers)}")
print(f" Trainable layers: {sum([1 for l in model.layers if l.trainable])}")
print(f" Non-trainable layers: {sum([1 for l in model.layers if not l.trainable])}")
  return model, base_model
def unfreeze_model(model, base_model, num_layers_to_unfreeze):
    """
    Unfreeze top layers of base model for fine-tuning
    Args:
      model: The full model
      base_model: The base pretrained model
      num_layers_to_unfreeze: Number of layers to unfreeze from the top
    """
base_model.trainable = True
# Freeze all layers except the last `num_layers_to_unfreeze`
for layer in base_model.layers[:-num_layers_to_unfreeze]:
layer.trainable = False
trainable_count = sum([1 for l in base_model.layers if l.trainable])
print(f"\n🔓 Unfroze {trainable_count} layers in base model for fine-tuning")
return model


 124
 125
 126
 127
 128
 129
 130
 131
 132
 133
 134
model = Model(inputs, outputs)
print(f"\n📊 Model Summary:")
print(f" Base Model: {model_name}")
print(f" Total layers: {len(model.layers)}")
print(f" Trainable layers: {sum([1 for l in model.layers if l.trainable])}")
print(f" Non-trainable layers: {sum([1 for l in model.layers if not l.trainable])}")
return model, base_model
def unfreeze_model(model, base_model, num_layers_to_unfreeze):
"""
Unfreeze top layers of base model for fine-tuning
Args:
model: The full model
base_model: The base pretrained model
num_layers_to_unfreeze: Number of layers to unfreeze from the top
"""
base_model.trainable = True
# Freeze all layers except the last `num_layers_to_unfreeze`
for layer in base_model.layers[:-num_layers_to_unfreeze]:
layer.trainable = False
trainable_count = sum([1 for l in base_model.layers if l.trainable])
print(f"\n🔓 Unfroze {trainable_count} layers in base model for fine-tuning")
return model
def get_optimizer(name, learning_rate):
"""Get optimizer by name"""
optimizers = {
'adam': Adam(learning_rate=learning_rate),
'sgd': SGD(learning_rate=learning_rate, momentum=0.9),
'adamax': Adamax(learning_rate=learning_rate),
'rmsprop': RMSprop(learning_rate=learning_rate)
}
return optimizers.get(name.lower(), Adam(learning_rate=learning_rate))
print("✅ Model factory loaded successfully")


In [ ]:
# ============================================================================
# 🔨 BUILD AND COMPILE MODEL
# ============================================================================
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
# Create model using factory
if MODEL_NAME == 'AlexNet':
    model = AlexNet(input_shape, NUM_CLASSES)
base_model = None
else:
     model, base_model = create_transfer_model(
      MODEL_NAME,
      input_shape,
      NUM_CLASSES,
      freeze_base=FREEZE_BASE
  )
# Compile model
optimizer = get_optimizer(OPTIMIZER, LEARNING_RATE)
model.compile(
optimizer=optimizer,
loss='categorical_crossentropy',
metrics=[
'accuracy',
tf.keras.metrics.Precision(name='precision'),
tf.keras.metrics.Recall(name='recall'),
tf.keras.metrics.AUC(name='auc')
]
)
print(f"\n🎯 Model: {MODEL_NAME}")
print(f"📊 Optimizer: {OPTIMIZER} (lr={LEARNING_RATE})")
print(f"📁 Classes: {NUM_CLASSES}")
model.summary()


In [ ]:
# ============================================================================
# 📞 CALLBACKS SETUP
# ============================================================================
callbacks = []
if USE_EARLY_STOPPING:
early_stopping = tf.keras.callbacks.EarlyStopping(
monitor='val_loss',
patience=EARLY_STOPPING_PATIENCE,
restore_best_weights=True,
verbose=1
)
callbacks.append(early_stopping)
print("✅ Early stopping enabled")
if USE_LR_SCHEDULER:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
monitor='val_loss',
factor=0.2,
patience=5,
min_lr=1e-7,
verbose=1
)
callbacks.append(lr_scheduler)
print("✅ Learning rate scheduler enabled")
if USE_MODEL_CHECKPOINT:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
MODEL_SAVE_PATH,
monitor='val_accuracy',
save_best_only=True,
verbose=1
)
callbacks.append(checkpoint)
print(f"✅ Model checkpoint enabled -> {MODEL_SAVE_PATH}")
# TensorBoard logging
log_dir = f"logs/{MODEL_NAME}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
callbacks.append(tensorboard_callback)
print(f"✅ TensorBoard logging -> {log_dir}")


In [ ]:
# ============================================================================
# 🚀 PHASE 1: INITIAL TRAINING (with frozen base)
# ============================================================================
print(f"\n{'='*60}")
print(f"🚀 Starting Phase 1 Training: {MODEL_NAME}")
print(f"{'='*60}")
history = model.fit(
train,
validation_data=valid,
epochs=EPOCHS,
verbose=1,
callbacks=callbacks,
)
print(f"\n✅ Phase 1 training complete!")

In [ ]:
# ============================================================================
# 🔧 PHASE 2: FINE-TUNING (unfreeze top layers)
# ============================================================================
if USE_FINE_TUNING and base_model is not None:
print(f"\n{'='*60}")
print(f"🔧 Starting Phase 2: Fine-tuning {MODEL_NAME}")
print(f"{'='*60}")
# Unfreeze top layers
model = unfreeze_model(model, base_model, UNFREEZE_LAYERS)
# Recompile with lower learning rate
model.compile(
optimizer=get_optimizer(OPTIMIZER, FINE_TUNE_LR),
loss='categorical_crossentropy',
metrics=[
'accuracy',
tf.keras.metrics.Precision(name='precision'),
tf.keras.metrics.Recall(name='recall'),
tf.keras.metrics.AUC(name='auc')
]
)
# Continue training
history_fine = model.fit(
train,
validation_data=valid,
epochs=FINE_TUNE_EPOCHS,
verbose=1,
callbacks=callbacks,
)
# Merge histories
for key in history.history:
history.history[key].extend(history_fine.history[key])
print(f"\n✅ Fine-tuning complete!")
else:
if MODEL_NAME == 'AlexNet':
print("ℹ️ AlexNet doesn't support fine-tuning (not a pretrained model)")
else:
print("ℹ️ Fine-tuning disabled in configuration")


In [ ]:
# ============================================================================
# 📊 TRAINING VISUALIZATION
# ============================================================================
def plot_training_history(history, model_name):
"""Plot training metrics"""
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].set_title(f'{model_name} - Accuracy', fontsize=14)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
# Loss
axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title(f'{model_name} - Loss', fontsize=14)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
# Precision
axes[1, 0].plot(history.history['precision'], label='Train', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Validation', linewidth=2)
axes[1, 0].set_title(f'{model_name} - Precision', fontsize=14)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
# Recall
axes[1, 1].plot(history.history['recall'], label='Train', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Validation', linewidth=2)
axes[1, 1].set_title(f'{model_name} - Recall', fontsize=14)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{model_name}_training_history.png', dpi=150)
plt.show()
plot_training_history(history, MODEL_NAME)

In [ ]:
# ============================================================================
# 🎯 MODEL EVALUATION
# ============================================================================
# Evaluate on test set
print(f"\n{'='*60}")
print(f"📊 Evaluating {MODEL_NAME} on Test Set")
print(f"{'='*60}")
# Reset generator
valid.reset()
# Evaluate
results = model.evaluate(valid, verbose=1)
print(f"\n📈 Final Results:")
print(f" Loss: {results[0]:.4f}")
print(f" Accuracy: {results[1]*100:.2f}%")
print(f" Precision: {results[2]*100:.2f}%")
print(f" Recall: {results[3]*100:.2f}%")
print(f" AUC: {results[4]*100:.2f}%")
# Calculate F1 Score
f1_score = 2 * (results[2] * results[3]) / (results[2] + results[3] + 1e-7)
print(f" F1 Score: {f1_score*100:.2f}%")


In [ ]:
# ============================================================================
# 🔢 CONFUSION MATRIX & CLASSIFICATION REPORT
# ============================================================================
# Get predictions
valid.reset()
predictions = model.predict(valid, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = valid.classes
# Get class names
class_names = list(valid.class_indices.keys())
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
xticklabels=class_names, yticklabels=class_names)
plt.title(f'{MODEL_NAME} - Confusion Matrix', fontsize=16)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_confusion_matrix.png', dpi=150)
plt.show()
# Classification Report
print(f"\n{'='*60}")
print(f"📋 Classification Report - {MODEL_NAME}")
print(f"{'='*60}\n")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ============================================================================
# 💾 SAVE MODEL
# ============================================================================
# Save the model
model.save(MODEL_SAVE_PATH)
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")
# Save training history (convert numpy float32 to Python float for JSON serialization)
import json
history_path = f'{MODEL_NAME}_training_history.json'
# Convert numpy arrays/floats to Python native types
history_dict = {key: [float(val) for val in values] for key, values in history.history.items()}
with open(history_path, 'w') as f:
json.dump(history_dict, f)
print(f"✅ Training history saved to: {history_path}")
# Save model configuration
config = {
'model_name': MODEL_NAME,
'input_shape': (IMG_HEIGHT, IMG_WIDTH, 3),
'num_classes': NUM_CLASSES,
'class_names': class_names,
'batch_size': BATCH_SIZE,
'optimizer': OPTIMIZER,
'learning_rate': LEARNING_RATE,
'final_accuracy': float(results[1]),
'final_loss': float(results[0])
}
config_path = f'{MODEL_NAME}_config.json'
with open(config_path, 'w') as f:
json.dump(config, f, indent=2)
print(f"✅ Model config saved to: {config_path}")

In [ ]:
# ============================================================================
# 🎥 REAL-TIME WEBCAM EMOTION DETECTION
# ============================================================================
def run_realtime_detection(model, class_names, input_size):
"""
Run real-time emotion detection using webcam
Args:
model: Trained Keras model
class_names: List of emotion class names
input_size: Tuple (height, width) for model input
"""
# Load face detector
face_cascade = cv2.CascadeClassifier(
cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
# Emotion colors (BGR format)
emotion_colors = {
'angry': (0, 0, 255), # Red
'disgust': (0, 128, 0), # Dark Green
'fear': (128, 0, 128), # Purple
'happy': (0, 255, 255), # Yellow
'neutral': (200, 200, 200), # Gray
'sad': (255, 0, 0), # Blue
'surprise': (0, 165, 255) # Orange
}
cap = cv2.VideoCapture(0)
if not cap.isOpened():
print("❌ Error: Could not open webcam")
return
print("🎥 Starting real-time detection... Press 'q' to quit")
while True:
ret, frame = cap.read()
if not ret:
break
# Convert to grayscale for face detection
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
# Detect faces
faces = face_cascade.detectMultiScale(gray, 1.3, 5, minSize=(30, 30))
for (x, y, w, h) in faces:
# Extract and preprocess face
face = frame[y:y+h, x:x+w]
face_resized = cv2.resize(face, input_size)
face_normalized = face_resized / 255.0
face_input = np.expand_dims(face_normalized, 0)
# Predict emotion
predictions = model.predict(face_input, verbose=0)
emotion_idx = np.argmax(predictions[0])
emotion = class_names[emotion_idx]
confidence = predictions[0][emotion_idx] * 100
# Get color for emotion
color = emotion_colors.get(emotion.lower(), (0, 255, 0))
# Draw rectangle and label
cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
label = f"{emotion}: {confidence:.1f}%"
label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
# Background for label
cv2.rectangle(frame, (x, y-label_size[1]-10),
(x+label_size[0], y), color, -1)
cv2.putText(frame, label, (x, y-5),
cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
# Display frame
cv2.imshow(f'Emotion Detection - {MODEL_NAME}', frame)
# Exit on 'q' key
if cv2.waitKey(1) & 0xFF == ord('q'):
break
cap.release()
cv2.destroyAllWindows()
print("✅ Real-time detection stopped")
# Run detection (uncomment to use)
# run_realtime_detection(model, class_names, (IMG_WIDTH, IMG_HEIGHT))

In [ ]:
# ============================================================================
# 🔄 MODEL COMPARISON UTILITY
# ============================================================================
def compare_models(model_results):
"""
Compare results from multiple models
Args:
model_results: Dict with model names as keys and metrics dict as values
Example: {'VGG16': {'accuracy': 0.85, 'loss': 0.4}, ...}
"""
models = list(model_results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, metric in enumerate(metrics):
values = [model_results[m].get(metric, 0) * 100 for m in models]
bars = axes[i].bar(models, values, color=plt.cm.viridis(np.linspace(0, 1, len(models))))
axes[i].set_title(metric.replace('_', ' ').title(), fontsize=14)
axes[i].set_ylabel('Percentage')
axes[i].set_ylim(0, 100)
axes[i].tick_params(axis='x', rotation=45)
# Add value labels on bars
for bar, val in zip(bars, values):
axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
# Example usage (uncomment after training multiple models):
# model_comparison_results = {
# 'AlexNet': {'accuracy': 0.65, 'precision': 0.62, 'recall': 0.60, 'f1_score': 0.61},
# 'MobileNetV2': {'accuracy': 0.78, 'precision': 0.76, 'recall': 0.75, 'f1_score': 0.755},
# 'ResNet50': {'accuracy': 0.82, 'precision': 0.80, 'recall': 0.79, 'f1_score': 0.795},
# 'EfficientNetB0': {'accuracy': 0.85, 'precision': 0.83, 'recall': 0.82, 'f1_score': 0.825},
# }
# compare_models(model_comparison_results)
print("📊 Model comparison utility ready - train multiple models and compare!")